In [ ]:
import pandas as pd
import numpy as np

In [ ]:
student_timeline_part1 = pd.read_csv(
    "../data/processed/student_timeline_part1.csv"
)

vle = pd.read_csv("../../dataset/raw/vle.csv")
studentVle = pd.read_csv("../../dataset/raw/studentVle.csv")

# 1) Cleaning Data

## 1.1 Student VLE - Cleaning

In [ ]:
studentVle.info()
studentVle

In [ ]:
print("sum_click null values:", studentVle["sum_click"].isna().sum())
print("sum_click negative values:", (studentVle["sum_click"] < 0).sum())
print("date null values:", studentVle["date"].isna().sum())

## 1.2 VLE - Cleaning

In [ ]:
vle.info()
vle

In [ ]:
print("VLE key duplicates:", vle.duplicated(
    subset=["code_module", "code_presentation", "id_site"]
).sum())

# 2) Merging Data

## 2.1 Merging Student VLE with VLE

In [ ]:
vle_activity_df = studentVle.merge(
    vle,
    on=["code_module", "code_presentation", "id_site"],
    how="left",
    validate="many_to_one"
)

print("studentVle rows:", len(studentVle))
print("merged rows:", len(vle_activity_df))
print("activity_type null values:", vle_activity_df["activity_type"].isna().sum())

In [ ]:
vle_activity_df = vle_activity_df.sort_values(
    ["id_student", "code_module", "code_presentation", "date"]
).reset_index(drop=True)

sample = vle_activity_df[
    (vle_activity_df["id_student"] == 6516) &
    (vle_activity_df["code_module"] == "AAA") &
    (vle_activity_df["code_presentation"] == "2014J")
]

sample[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date",
        "id_site",
        "activity_type",
        "sum_click"
    ]
].head(30)

# 3) Preparing VLE Data

## 3.1 Removing Pre-Module Activity

In [ ]:
vle_activity_df = vle_activity_df[
    vle_activity_df["date"] >= 0
].copy()

## 3.2 Daily Student Activity

In [ ]:
group_cols = [
    "id_student",
    "code_module",
    "code_presentation",
]

daily_group_cols = group_cols + ["date"]

vle_daily = (
    vle_activity_df
    .groupby(daily_group_cols, as_index=False)["sum_click"]
    .sum()
    .rename(columns={"sum_click": "daily_clicks"})
)

vle_daily = vle_daily.sort_values(
    group_cols + ["date"]
).reset_index(drop=True)

vle_daily

## 3.3 Cumulative Clicks

In [ ]:
vle_daily["cumulative_clicks"] = (
    vle_daily
    .groupby(group_cols)["daily_clicks"]
    .cumsum()
)

vle_daily

# 4) Aligning VLE Activity with Assessment Checkpoints

In [ ]:
checkpoint_keys = [
    "id_student",
    "code_module",
    "code_presentation",
    "id_assessment",
]

checkpoint_features = student_timeline_part1[
    checkpoint_keys + ["time_step", "date"]
].copy()

In [ ]:
vle_for_merge = vle_daily.rename(
    columns={"date": "vle_date"}
)

checkpoint_lookup = checkpoint_features.sort_values(
    ["date"] + group_cols
)

vle_for_merge = vle_for_merge.sort_values(
    ["vle_date"] + group_cols
)

aligned = pd.merge_asof(
    checkpoint_lookup,
    vle_for_merge[
        group_cols + ["vle_date", "cumulative_clicks"]
    ],
    left_on="date",
    right_on="vle_date",
    by=group_cols,
    direction="backward",
    allow_exact_matches=True,
)

aligned["cumulative_clicks"] = (
    aligned["cumulative_clicks"].fillna(0)
)
aligned

## 4.1 Checking the Alignment

In [ ]:
sample = aligned[
    (aligned["id_student"] == 6516) &
    (aligned["code_module"] == "AAA") &
    (aligned["code_presentation"] == "2014J")
][
    ["time_step", "date", "vle_date", "cumulative_clicks"]
]

print(sample)

# 5) Activity Type Features

## 5.1 Daily Activity by Type

In [ ]:
type_group_cols = [
    "id_student",
    "code_module",
    "code_presentation",
    "activity_type",
    "date",
]

vle_type_daily = (
    vle_activity_df
    .groupby(type_group_cols, as_index=False)["sum_click"]
    .sum()
    .rename(columns={"sum_click": "daily_clicks"})
)

## 5.2 Cumulative Activity by Type

In [ ]:
group_cols_type = [
    "id_student",
    "code_module",
    "code_presentation",
    "activity_type",
]

vle_type_daily = vle_type_daily.sort_values(
    group_cols_type + ["date"]
).reset_index(drop=True)

vle_type_daily["cumulative_clicks"] = (
    vle_type_daily
    .groupby(group_cols_type)["daily_clicks"]
    .cumsum()
)

activity_types = sorted(
    vle_type_daily["activity_type"]
    .dropna()
    .unique()
)

print("Activity types:", activity_types)

## 5.3 Aligning Activity Types with Checkpoints

In [ ]:
behavior_features = checkpoint_features.copy()

for activity_type in activity_types:
    type_df = vle_type_daily[
        vle_type_daily["activity_type"] == activity_type
    ].copy()

    type_df = type_df.sort_values(
        ["date"] + group_cols
    )

    aligned_type = pd.merge_asof(
        checkpoint_features.sort_values(
            ["date"] + group_cols
        ),
        type_df[
            group_cols + ["date", "cumulative_clicks"]
        ],
        on="date",
        by=group_cols,
        direction="backward",
        allow_exact_matches=True,
    )

    column_name = f"{activity_type}_cumulative_clicks"

    behavior_features[column_name] = (
        aligned_type["cumulative_clicks"]
        .fillna(0)
    )

activity_feature_cols = [
    col for col in behavior_features.columns
    if col.endswith("_cumulative_clicks")
]

behavior_features

## 5.4 Checking Activity Type Totals

In [ ]:
check_sum = behavior_features[activity_feature_cols].sum(axis=1)

print(
    "Maximum absolute difference:",
    (check_sum - aligned["cumulative_clicks"]).abs().max()
)

print(
    "Activity feature null values:",
    behavior_features[activity_feature_cols].isna().sum().sum()
)

# 6) Engagement Features

## 6.1 Active Days

In [ ]:
vle_active = vle_daily[
    vle_daily["daily_clicks"] > 0
].copy()

vle_active["active_days"] = (
    vle_active
    .groupby(group_cols)
    .cumcount() + 1
)

vle_active = vle_active.rename(
    columns={"date": "vle_date"}
)

## 6.2 Aligning Active Days

In [ ]:
active_lookup = checkpoint_features.sort_values(
    ["date"] + group_cols
)

vle_active = vle_active.sort_values(
    ["vle_date"] + group_cols
)

active_aligned = pd.merge_asof(
    active_lookup,
    vle_active[
        group_cols + ["vle_date", "active_days"]
    ],
    left_on="date",
    right_on="vle_date",
    by=group_cols,
    direction="backward",
    allow_exact_matches=True,
)

active_aligned["days_since_last_activity"] = (
    active_aligned["date"] - active_aligned["vle_date"]
)

active_aligned["has_activity"] = (
    active_aligned["vle_date"].notna().astype("int8")
)

active_aligned["active_days"] = (
    active_aligned["active_days"].fillna(0)
)

active_aligned

# 7) Recent Activity

Recent activity is calculated from cumulative clicks.

In [ ]:
def calculate_recent_clicks(
    checkpoint_df,
    vle_daily,
    window_days,
    group_cols,
):
    checkpoints = checkpoint_df[
        group_cols + ["id_assessment", "date"]
    ].copy()

    checkpoints["_checkpoint_id"] = np.arange(
        len(checkpoints)
    )

    end_lookup = checkpoints.sort_values(
        ["date"] + group_cols
    )

    vle_end = vle_daily.sort_values(
        ["date"] + group_cols
    )

    end_aligned = pd.merge_asof(
        end_lookup,
        vle_end[
            group_cols + ["date", "cumulative_clicks"]
        ],
        on="date",
        by=group_cols,
        direction="backward",
        allow_exact_matches=True,
    )

    end_values = end_aligned[
        ["_checkpoint_id", "cumulative_clicks"]
    ].rename(
        columns={"cumulative_clicks": "end_cumulative"}
    )

    start_lookup = checkpoints.copy()
    start_lookup["lookup_date"] = (
        start_lookup["date"] - window_days
    )

    start_lookup = start_lookup.sort_values(
        ["lookup_date"] + group_cols
    )

    vle_start = vle_daily.sort_values(
        ["date"] + group_cols
    )

    start_aligned = pd.merge_asof(
        start_lookup,
        vle_start[
            group_cols + ["date", "cumulative_clicks"]
        ],
        left_on="lookup_date",
        right_on="date",
        by=group_cols,
        direction="backward",
        allow_exact_matches=True,
    )

    start_values = start_aligned[
        ["_checkpoint_id", "cumulative_clicks"]
    ].rename(
        columns={"cumulative_clicks": "start_cumulative"}
    )

    result = end_values.merge(
        start_values,
        on="_checkpoint_id",
        how="left",
        validate="one_to_one",
    )

    result["end_cumulative"] = (
        result["end_cumulative"].fillna(0)
    )

    result["start_cumulative"] = (
        result["start_cumulative"].fillna(0)
    )

    result[f"recent_{window_days}d_clicks"] = (
        result["end_cumulative"]
        - result["start_cumulative"]
    )

    return checkpoints[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "id_assessment",
            "_checkpoint_id",
        ]
    ].merge(
        result[
            [
                "_checkpoint_id",
                f"recent_{window_days}d_clicks",
            ]
        ],
        on="_checkpoint_id",
        how="left",
        validate="one_to_one",
    ).drop(
        columns="_checkpoint_id"
    )

In [ ]:
recent_7 = calculate_recent_clicks(
    checkpoint_features,
    vle_daily,
    window_days=7,
    group_cols=group_cols,
)

recent_14 = calculate_recent_clicks(
    checkpoint_features,
    vle_daily,
    window_days=14,
    group_cols=group_cols,
)

## 7.1 Merging Engagement Features

In [ ]:
behavior_features = active_aligned[
    checkpoint_keys +
    [
        "active_days",
        "days_since_last_activity",
        "has_activity",
    ]
].copy()

behavior_features = behavior_features.merge(
    recent_7,
    on=checkpoint_keys,
    how="left",
    validate="one_to_one",
)

behavior_features = behavior_features.merge(
    recent_14,
    on=checkpoint_keys,
    how="left",
    validate="one_to_one",
)

behavior_features

# 8) Building Student Timeline Part 2

In [ ]:
student_timeline_part2 = student_timeline_part1.merge(
    aligned[
        checkpoint_keys + ["cumulative_clicks"]
    ],
    on=checkpoint_keys,
    how="left",
    validate="one_to_one",
)

student_timeline_part2 = student_timeline_part2.merge(
    behavior_features[
        checkpoint_keys +
        activity_feature_cols +
        [
            "active_days",
            "days_since_last_activity",
            "has_activity",
            "recent_7d_clicks",
            "recent_14d_clicks",
        ]
    ],
    on=checkpoint_keys,
    how="left",
    validate="one_to_one",
)

student_timeline_part2.info()

## 8.1 Score Availability

In [ ]:
student_timeline_part2["score_available"] = (
    student_timeline_part2["is_banked"].eq(1)
    |
    (
        student_timeline_part2["date_submitted"].ge(0)
        & student_timeline_part2["date_submitted"].le(
            student_timeline_part2["date"]
        )
    )
).astype("int8")

student_timeline_part2["score_observed"] = (
    student_timeline_part2["score"]
    .where(student_timeline_part2["score_available"] == 1)
)

student_timeline_part2["submission_offset_days"] = (
    student_timeline_part2["date_submitted"]
    - student_timeline_part2["date"]
).where(
    student_timeline_part2["score_available"] == 1
)

student_timeline_part2[[
    "date", "date_submitted", "score",
    "score_available", "score_observed",
    "submission_offset_days"
]].head(20)

## 8.2 Target Column

In [ ]:
AT_RISK_RESULTS = {
    "Fail",
    # "Withdrawn",
}

student_timeline_part2["target"] = (
    student_timeline_part2["final_result"]
    .isin(AT_RISK_RESULTS)
    .astype("int8")
)

print(student_timeline_part2["target"].value_counts())

print(
    "Sequences with multiple targets:",
    student_timeline_part2
    .groupby(["id_student", "code_module", "code_presentation"])["target"]
    .nunique()
    .gt(1)
    .sum()
)

## 8.3 Sequence Information

In [ ]:
sequence_group_cols = [
    "id_student",
    "code_module",
    "code_presentation",
]

student_timeline_part2 = student_timeline_part2.sort_values(
    sequence_group_cols + ["time_step"]
).reset_index(drop=True)

student_timeline_part2["sequence_id"] = (
    student_timeline_part2
    .groupby(sequence_group_cols)
    .ngroup()
)

student_timeline_part2["sequence_length"] = (
    student_timeline_part2
    .groupby("sequence_id")["time_step"]
    .transform("max")
)

# 9) Final Data Validation

In [ ]:
activity_feature_cols = [
    col for col in student_timeline_part2.columns
    if col.endswith("_cumulative_clicks")
    and col != "cumulative_clicks"
]

print("Duplicate checkpoints:", student_timeline_part2.duplicated(
    subset=checkpoint_keys
).sum())

print("Bad time order:", (
    student_timeline_part2
    .sort_values(sequence_group_cols + ["time_step"])
    .groupby("sequence_id")["time_step"]
    .apply(lambda x: not x.is_monotonic_increasing)
    .sum()
))

print("Negative delta_days:", (student_timeline_part2["delta_days"] < 0).sum())
print("module_progress outside [0,1]:", (
    (student_timeline_part2["module_progress"] < 0)
    | (student_timeline_part2["module_progress"] > 1)
).sum())

print("recent7 > recent14:", (
    student_timeline_part2["recent_7d_clicks"]
    > student_timeline_part2["recent_14d_clicks"]
).sum())

print("recent14 > cumulative:", (
    student_timeline_part2["recent_14d_clicks"]
    > student_timeline_part2["cumulative_clicks"]
).sum())

print("negative recent7:", (student_timeline_part2["recent_7d_clicks"] < 0).sum())
print("negative recent14:", (student_timeline_part2["recent_14d_clicks"] < 0).sum())
print("negative days_since:", (
    student_timeline_part2["days_since_last_activity"].dropna() < 0
).sum())

In [ ]:
sequence_lengths = student_timeline_part2.groupby("sequence_id").size()
print(sequence_lengths.describe())
print(sequence_lengths.value_counts().sort_index())

## 9.1 Example Checkpoint

In [ ]:
sample = student_timeline_part2[
    (student_timeline_part2["id_student"] == 6516) &
    (student_timeline_part2["code_module"] == "AAA") &
    (student_timeline_part2["code_presentation"] == "2014J")
][
    [
        "time_step",
        "date",
        "cumulative_clicks",
        "active_days",
        "days_since_last_activity",
        "has_activity",
        "recent_7d_clicks",
        "recent_14d_clicks",
    ]
]

print(sample)

# 10) Exporting CSV

`student_timeline_part2.csv` is the final timeline dataset before ML preprocessing.

In [ ]:
student_timeline_part2.to_csv(
    "../data/processed/student_timeline_part2.csv",
    index=False,
)